<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes./blob/main/M%C3%B3dulo_de_pruebas_unitarias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import unittest
from aerocargo_matrix import (
    validar_matrices,
    calcular_ocupacion,
    evaluar_balance,
    extraer_submatriz_critica,
)


class TestValidarMatrices(unittest.TestCase):

    def test_caso_tipico_3x4(self):
        cargas = [[187, 243, 96, 310], [402, 88, 175, 260], [59, 340, 210, 77]]
        capacidades = [[400, 400, 400, 400], [500, 500, 500, 500], [350, 350, 350, 350]]
        self.assertTrue(validar_matrices(cargas, capacidades))

    def test_bahia_minima_2x2(self):
        cargas = [[0, 0], [0, 0]]
        capacidades = [[275, 275], [275, 275]]
        self.assertTrue(validar_matrices(cargas, capacidades))

    def test_columnas_no_coinciden_entre_matrices(self):
        cargas = [[187, 243, 96], [402, 88, 175]]
        capacidades = [[400, 400, 400, 400], [500, 500, 500, 500]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_una_sola_fila_no_es_valido(self):
        cargas = [[187, 243, 96, 310]]
        capacidades = [[400, 400, 400, 400]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_una_sola_columna_no_es_valido(self):
        cargas = [[187], [402], [59]]
        capacidades = [[400], [500], [350]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_fila_con_longitud_distinta(self):
        cargas = [[187, 243, 96, 310], [402, 88, 175]]
        capacidades = [[400, 400, 400, 400], [500, 500, 500, 500]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_peso_negativo_no_permitido(self):
        cargas = [[187, -15, 96, 310], [402, 88, 175, 260]]
        capacidades = [[400, 400, 400, 400], [500, 500, 500, 500]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_capacidad_en_cero_no_permitida(self):
        cargas = [[187, 243, 96, 310], [402, 88, 175, 260]]
        capacidades = [[400, 0, 400, 400], [500, 500, 500, 500]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_capacidad_negativa_no_permitida(self):
        cargas = [[187, 243, 96, 310], [402, 88, 175, 260]]
        capacidades = [[400, -50, 400, 400], [500, 500, 500, 500]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_celda_con_peso_cero_es_valida(self):
        cargas = [[0, 243, 96, 310], [402, 88, 175, 260]]
        capacidades = [[400, 400, 400, 400], [500, 500, 500, 500]]
        self.assertTrue(validar_matrices(cargas, capacidades))


class TestCalcularOcupacion(unittest.TestCase):

    def test_calculo_de_porcentajes(self):
        cargas = [[176, 264], [90, 405]]
        capacidades = [[220, 330], [180, 450]]
        resultado = calcular_ocupacion(cargas, capacidades)
        esperado = [[80.0, 80.0], [50.0, 90.0]]
        self.assertEqual(resultado["matriz_porcentajes"], esperado)

    def test_identifica_celdas_sobrecargadas(self):
        cargas = [[318, 140, 205], [96, 460, 130]]
        capacidades = [[300, 300, 300], [300, 400, 300]]
        resultado = calcular_ocupacion(cargas, capacidades)
        self.assertIn((0, 0), resultado["celdas_sobrecargadas"])
        self.assertIn((1, 1), resultado["celdas_sobrecargadas"])
        self.assertEqual(len(resultado["celdas_sobrecargadas"]), 2)

    def test_bahia_sin_ninguna_sobrecarga(self):
        cargas = [[150, 220, 90], [310, 180, 275]]
        capacidades = [[400, 400, 400], [400, 400, 400]]
        resultado = calcular_ocupacion(cargas, capacidades)
        self.assertEqual(resultado["celdas_sobrecargadas"], [])

    def test_celda_exactamente_en_el_limite_no_es_sobrecarga(self):
        cargas = [[300, 150], [150, 300]]
        capacidades = [[300, 300], [300, 300]]
        resultado = calcular_ocupacion(cargas, capacidades)
        self.assertEqual(resultado["celdas_sobrecargadas"], [])

    def test_matrices_de_entrada_quedan_intactas(self):
        cargas = [[176, 264], [90, 405]]
        capacidades = [[220, 330], [180, 450]]
        respaldo_cargas = [fila[:] for fila in cargas]
        respaldo_capacidades = [fila[:] for fila in capacidades]

        calcular_ocupacion(cargas, capacidades)

        self.assertEqual(cargas, respaldo_cargas)
        self.assertEqual(capacidades, respaldo_capacidades)


class TestEvaluarBalance(unittest.TestCase):

    def test_bahia_de_4_columnas_perfectamente_balanceada(self):
        cargas = [[210, 95, 95, 210], [180, 60, 60, 180]]
        resultado = evaluar_balance(cargas, tolerancia=0.0)
        self.assertEqual(resultado["desbalance_lateral"], 0.0)
        self.assertTrue(resultado["balance_aprobado"])

    def test_bahia_de_4_columnas_fuera_de_tolerancia(self):
        cargas = [[430, 310, 90, 60]]
        resultado = evaluar_balance(cargas, tolerancia=200.0)
        self.assertEqual(resultado["desbalance_lateral"], 590.0)
        self.assertFalse(resultado["balance_aprobado"])

    def test_bahia_de_5_columnas_omite_la_columna_del_medio(self):
        # col 0 y 1 = izquierda, col 2 = eje central (no cuenta),
        # col 3 y 4 = derecha
        cargas = [[80, 70, 999, 65, 85]]
        resultado = evaluar_balance(cargas, tolerancia=0.0)
        self.assertEqual(resultado["desbalance_lateral"], 0.0)
        self.assertTrue(resultado["balance_aprobado"])

    def test_vector_de_peso_por_fila(self):
        cargas = [[45, 60, 33], [12, 8, 19], [100, 5, 5]]
        resultado = evaluar_balance(cargas, tolerancia=500.0)
        self.assertEqual(resultado["pesos_por_fila"], [138, 39, 110])

    def test_desbalance_justo_en_el_borde_de_tolerancia(self):
        cargas = [[210, 15]]
        resultado = evaluar_balance(cargas, tolerancia=195.0)
        self.assertTrue(resultado["balance_aprobado"])

    def test_desbalance_un_kg_arriba_de_tolerancia(self):
        cargas = [[210, 15]]
        resultado = evaluar_balance(cargas, tolerancia=194.0)
        self.assertFalse(resultado["balance_aprobado"])


class TestExtraerSubmatrizCritica(unittest.TestCase):

    def test_localiza_la_zona_de_mayor_ocupacion(self):
        matriz_porcentajes = [
            [40, 55, 190, 205],
            [35, 60, 210, 195],
            [20, 15, 30, 25],
        ]
        submatriz = extraer_submatriz_critica(matriz_porcentajes, 2, 2)
        self.assertEqual(submatriz, [[190, 205], [210, 195]])

    def test_ventana_mas_grande_que_la_bahia_retorna_none(self):
        matriz_porcentajes = [[65, 80, 95], [70, 85, 60]]
        self.assertIsNone(extraer_submatriz_critica(matriz_porcentajes, 5, 2))

    def test_ventana_con_dimensiones_en_cero_retorna_none(self):
        matriz_porcentajes = [[65, 80], [70, 85]]
        self.assertIsNone(extraer_submatriz_critica(matriz_porcentajes, 0, 2))

    def test_ventana_del_tamano_completo_de_la_bahia(self):
        matriz_porcentajes = [[62, 71], [88, 45]]
        submatriz = extraer_submatriz_critica(matriz_porcentajes, 2, 2)
        self.assertEqual(submatriz, [[62, 71], [88, 45]])

    def test_desempate_por_mayor_cantidad_de_celdas_sobrecargadas(self):
        # ambas ventanas 1x2 tienen promedio 150, pero la segunda
        # tiene las dos celdas sobrecargadas y la primera solo una
        matriz_porcentajes = [[250, 50, 190, 110]]
        submatriz = extraer_submatriz_critica(matriz_porcentajes, 1, 2)
        self.assertEqual(submatriz, [[190, 110]])

    def test_no_se_modifica_la_matriz_de_porcentajes_original(self):
        matriz_porcentajes = [[80, 95, 205], [60, 70, 190]]
        respaldo = [fila[:] for fila in matriz_porcentajes]
        extraer_submatriz_critica(matriz_porcentajes, 2, 1)
        self.assertEqual(matriz_porcentajes, respaldo)


if __name__ == "__main__":
    unittest.main(verbosity=2)